# Exploratory Data Analysis | FD001 - C-MAPSS simulation
We begin reading the FD001 training subset of the C-MAPPS simulation. Each file contains time-series measurements from 21 sensors along with three operating condition settings. The dataset has no header row, so we assign column names manually

1. Import Libraries and Project Modules

In [ ]:
# standard
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# project
from xai_aviation_rul.data_loader import load_cmapss
from xai_aviation_rul.preprocessor import compute_rul, drop_constant_sensors, normalize, get_last_cycle
from xai_aviation_rul.visualizer import (
    plot_rul_distribution,
    plot_sensor_trends,
    plot_sensor_variance,
    plot_correlation_heatmap,
    save_table_as_figure,
)
FIG_DIR = Path("figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

: 

In [ ]:
# load dataset
df_train = load_cmapss(fd=1, subset="train")
df_test  = load_cmapss(fd=1, subset="test")

In [ ]:
# show head and shapes
display(df_train.head())
print(f"Training set shape: {df_train.shape}")
print(f"Test set shape: {df_test.shape}")

2. Dataset Statistics

In [ ]:
# dataset statistics table
# Dataset Statistics Table
cycles_per_engine = df_train.groupby("unit_number")["time_in_cycles"].max()

stats_table = pd.DataFrame([{
    "Engines (Train)": int(df_train["unit_number"].nunique()),
    "Engines (Test)": int(df_test["unit_number"].nunique()),
    "Total Training Rows": int(len(df_train)),
    "Total Test Rows": int(len(df_test)),
    "Sensors (Raw)": 21,
    "Operating Settings": 3,
    "Min Cycles (Train)": int(cycles_per_engine.min()),
    "Max Cycles (Train)": int(cycles_per_engine.max()),
    "Mean Cycles (Train)": round(float(cycles_per_engine.mean()), 1),
    "RUL Cap Applied": 125,
}]).T.rename(columns={0: "Value"}).rename_axis("Metric")

save_table_as_figure(
    stats_table,
    save_path=FIG_DIR / "table1_dataset_statistics.png",
    title="Table 1 — Dataset Statistics"
)

3. Compute RUL

In [ ]:
# computer RUL
df_train = compute_rul(df_train, rul_cap=125)
display(df_train.loc[:, ["unit_number", "time_in_cycles", "RUL", "RUL_capped"]].head(5))

4. Sensor Variance Analysis

In [ ]:
# sensor variance analysis 
plot_sensor_variance(df_train, save_path=FIG_DIR / "fig_sensor_variance.png")

5. Drop Constant Sensors

In [ ]:
# drop constant sensors
before_sensors = [c for c in df_train.columns if c.startswith("sensor_")]

# call drop_constant_sensors — handle either return types (df or (df, dropped))
result = drop_constant_sensors(df_train, threshold=0.01)
if isinstance(result, tuple) and len(result) == 2:
    df_train, dropped_list = result
else:
    df_train = result
    after_sensors = [c for c in df_train.columns if c.startswith("sensor_")]
    dropped_list = sorted(set(before_sensors) - set(after_sensors))

remaining_sensors = sorted([c for c in df_train.columns if c.startswith("sensor_")])

print("Dropped sensors:", dropped_list)
print("Remaining sensors:", remaining_sensors)

6. Normalization

In [ ]:
# normalization
train_scaled, test_scaled, scaler = normalize(df_train.copy(), df_test.copy())
df_train = train_scaled
df_test = test_scaled
print("Normalization complete - scaler fitted on training sensors and applied to test.")

7. RUL Distribution

In [ ]:
# RUL distribution  
plot_rul_distribution(df_train, save_path=FIG_DIR / "fig_rul_distribution.png")

8. Sensor Degradation Trends

In [ ]:
# sensor degradation trends
sensor_cols_to_plot = ["sensor_2", "sensor_3", "sensor_4"]
plot_sensor_trends(df_train, engine_ids=[1, 2, 3], sensor_cols=sensor_cols_to_plot, save_path=FIG_DIR / "fig_sensor_trends.png")

9. Correlation Heatmap

In [ ]:
# correlation heatmap
plot_correlation_heatmap(df_train, save_path=FIG_DIR / "fig_correlation_heatmap.png")